In [555]:
import pyomo.environ as pyo
from queue import PriorityQueue
import time
import math
import matplotlib.pyplot as plt
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout
import numpy as np
import scipy.sparse

# --- 1. Problem Data ---
# Based on the table in the PDF
profits = {
    1: 22, 2: 18, 3: 35, 4: 14, 5: 60, 6: 12,
    7: 50, 8: 15, 9: 21, 10: 19, 11: 24, 12: 16
}
weights = {
    1: 5, 2: 6, 3: 8, 4: 4, 5: 11, 6: 3,
    7: 10, 8: 4, 9: 6, 10: 5, 11: 7, 12: 5
}
# Capacities
capacities = {1: 22, 2: 19}

# Item and Knapsack sets (using n=12)
ITEMS = list(profits.keys())
KNAPSACKS = list(capacities.keys())

In [556]:
def create_base_model():
    """Creates the base Pyomo model structure for the MKP."""
    model = pyo.ConcreteModel()
    
    # Sets
    model.I = pyo.Set(initialize=ITEMS)
    model.J = pyo.Set(initialize=KNAPSACKS)
    
    # Parameters
    model.p = pyo.Param(model.I, initialize=profits)
    model.w = pyo.Param(model.I, initialize=weights)
    model.C = pyo.Param(model.J, initialize=capacities)
    
    # Variables: x_ij
    # We define them as Reals between 0 and 1 for the LP relaxation
    model.x = pyo.Var(model.I, model.J, within=pyo.NonNegativeReals, bounds=(0, 1))
    
    # Objective: Maximize total profit
    def obj_rule(m):
        return sum(m.p[i] * m.x[i, j] for i in m.I for j in m.J)
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)
    
    # Constraint 1: Knapsack capacity
    def capacity_rule(m, j):
        return sum(m.w[i] * m.x[i, j] for i in m.I) <= m.C[j]
    model.capacity_con = pyo.Constraint(model.J, rule=capacity_rule)
    
    # Constraint 2: Each item in at most one knapsack
    def item_once_rule(m, i):
        return sum(m.x[i, j] for j in m.J) <= 1
    model.item_once_con = pyo.Constraint(model.I, rule=item_once_rule)
    
    return model

def solve_node_lp(model, solver, bounds_to_apply):
    # Apply the bounds for this specific node
    for (i, j), val in bounds_to_apply.items():
        model.x[i, j].fix(val)
        
    # Solve the LP with load_solutions=False to avoid warnings for infeasible cases
    try:
        results = solver.solve(model, tee=False, load_solutions=False)
    except Exception as e:
        print(f"Solver error: {e}")
        return 'error', None, None, False

    # --- Process Results ---
    
    # Check status
    status = results.solver.termination_condition
    if status == pyo.TerminationCondition.infeasible:
        return 'infeasible', None, None, False
    if status != pyo.TerminationCondition.optimal:
        # print(f"Warning: Node solve was not optimal. Status: {status}")
        return 'suboptimal', None, None, False

    # Load the solution only if optimal
    model.solutions.load_from(results)
    
    # Extract solution
    lp_value = model.obj()
    lp_solution = {}
    is_integer = True
    epsilon = 1e-6  # Tolerance for integrality

    for i in ITEMS:
        for j in KNAPSACKS:
            val = model.x[i, j].value
            if val is None:
                val = 0  # Handle non-assignment
            
            lp_solution[(i, j)] = val
            
            # Check for fractional values
            if abs(val - round(val)) > epsilon:
                is_integer = False
                
    return 'optimal', lp_value, lp_solution, is_integer

## Problem 2

In [557]:
def add_cut(model, min_cover_solution):
    """
    Adds a cover cut to the model.
    Expected input: min_cover_solution is a dictionary {key: 1/0} 
    derived from the minimal cover finder.
    """
    
    # 3. Ensure the model has a container for cuts
    # (It's better to add this in create_base_model, but this is a safety catch)
    if not hasattr(model, 'cuts'):
        model.cuts = pyo.ConstraintList()

    cut_expr = sum(model.x[k] for k in min_cover_solution) <= len(min_cover_solution) - 1
    
    # 5. Add to the list
    model.cuts.add(cut_expr)
    print(f"Added cut covering {len(min_cover_solution)} variables.")

In [558]:
solver = pyo.SolverFactory('gurobi_direct')
model = create_base_model()

In [559]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.20000000000002 optimal


{(1, 1): 0.0,
 (1, 2): 1.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.6,
 (10, 2): 0.2,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [560]:
min_cover1 = [(3,1),(5,1),(10,1)] 
add_cut(model, min_cover1)

Added cut covering 3 variables.


In [561]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 0.6,
 (1, 2): 0.3999999999999999,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [562]:
min_cover2 = [(3,1),(5,1),(1,1)] 
add_cut(model, min_cover2)

Added cut covering 3 variables.


In [563]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 1.0,
 (1, 2): 0.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.0,
 (3, 2): 1.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.6,
 (7, 2): 0.3999999999999999,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8000000000000002,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

## Problem 2.2


In [564]:
def sequential_lifting_cut(model, min_cover, solver=None):
    """
    Sequentially lift the base cover inequality for a single knapsack.
    Returns a dictionary alpha[(item, knapsack)] with the lifted coefficients.
    """
    if solver is None:
        solver = pyo.SolverFactory('gurobi_direct')

    knapsack = min_cover[0][1]
    cover = [i for (i, k) in min_cover if k == knapsack]
    rhs = len(cover) - 1

    alpha = {(i, knapsack): 1 if i in cover else 0 for i in ITEMS}

    order = (set(ITEMS) - set(cover))
    # Sort items by solution value (zeros first) then by weight
    order = sorted(list(order), key=lambda i: (model.x[i, knapsack].value if model.x[i, knapsack].value is not None else 0, weights[i]))
    
    for item in order: # Should be sorted more efficently
        residual_capacity = capacities[knapsack] - weights[item]
        if residual_capacity < 0:
            alpha[(item, knapsack)] = rhs
            continue

        sub_model = pyo.ConcreteModel()
        sub_model.COVER = pyo.Set(initialize=cover)
        sub_model.x = pyo.Var(sub_model.COVER, domain=pyo.Binary)

        def obj_rule(m):
            return sum(m.x[k] for k in m.COVER)
        sub_model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

        def capacity_rule(m):
            return sum(weights[k] * m.x[k] for k in m.COVER) <= residual_capacity
        sub_model.capacity = pyo.Constraint(rule=capacity_rule)

        results = solver.solve(sub_model, tee=False, load_solutions=False)
        if results.solver.termination_condition != pyo.TerminationCondition.optimal:
            raise RuntimeError("Sequential lifting subproblem failed to solve optimally.")

        sub_model.solutions.load_from(results)
        z_val = pyo.value(sub_model.obj)
        alpha[(item, knapsack)] = max(0, rhs - z_val)


    # Lets cut
    if not hasattr(model, 'cuts'):
        model.cuts = pyo.ConstraintList()
        
    cut_expr = sum(alpha[(i, knapsack)] * model.x[i, knapsack] for i in ITEMS) <= rhs
    model.cuts.add(cut_expr)



In [565]:
solver = pyo.SolverFactory('gurobi_direct')
model = create_base_model()

In [566]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.20000000000002 optimal


{(1, 1): 0.0,
 (1, 2): 1.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.6,
 (10, 2): 0.2,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [567]:
min_cover1 = [(3,1),(5,1),(10,1)] 
sequential_lifting_cut(model, min_cover1)

In [568]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 0.2,
 (1, 2): 0.8,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.0,
 (3, 2): 1.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 1.0,
 (7, 2): 0.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [569]:
#min_cover2 = [(1,1),(5,1),(1,1)] 
min_cover2 = [(1,2),(3,2),(6,2),(10,2)]
sequential_lifting_cut(model, min_cover2)

In [570]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.20000000000002 optimal


{(1, 1): 0.8,
 (1, 2): 0.2,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.6666666666666667,
 (3, 2): 0.3333333333333333,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.33333333333333326,
 (10, 2): 0.46666666666666673,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

## Problem 5

In [571]:
def create_fcnf_model():
    model = pyo.ConcreteModel(name="FCNF_Assignment3")

    # -------------------------------------------------------------------------
    # 1. Sets [cite: 139, 141]
    # -------------------------------------------------------------------------
    model.NODES = pyo.Set(initialize=['S', 'A', 'B', 'C', 'D', 'E'])
    model.ARCS = pyo.Set(dimen=2, initialize=[
        ('S', 'A'), ('S', 'B'), ('S', 'C'),
        ('A', 'B'), ('A', 'D'),
        ('B', 'C'), ('B', 'D'), ('B', 'E'),
        ('C', 'D'), ('C', 'E')
    ])

    # -------------------------------------------------------------------------
    # 2. Parameters [cite: 145-147, 151]
    # -------------------------------------------------------------------------
    # Net Supply/Demand (b_i): In - Out = b_i
    # Note: The problem defines positive b_i as demand and negative as supply.
    # S: -40, D: 25, E: 15, Others: 0
    b_data = {
        'S': -40,
        'A': 0,
        'B': 0,
        'C': 0,
        'D': 25,
        'E': 15
    }
    model.b = pyo.Param(model.NODES, initialize=b_data)

    # Arc Data: Variable Cost (c), Fixed Cost (f), Capacity (U)
    # Extracted from Table 1 
    arc_data = {
        ('S', 'A'): {'c': 2, 'f': 30, 'U': 25},
        ('S', 'B'): {'c': 3, 'f': 18, 'U': 20},
        ('S', 'C'): {'c': 4, 'f': 10, 'U': 20},
        ('A', 'B'): {'c': 1, 'f': 8,  'U': 15},
        ('B', 'C'): {'c': 1, 'f': 6,  'U': 15},
        ('A', 'D'): {'c': 3, 'f': 20, 'U': 20},
        ('B', 'D'): {'c': 2, 'f': 16, 'U': 25},
        ('B', 'E'): {'c': 2, 'f': 14, 'U': 15},
        ('C', 'D'): {'c': 1, 'f': 12, 'U': 15},
        ('C', 'E'): {'c': 3, 'f': 5,  'U': 20},
    }

    model.c = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['c'])
    model.f = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['f'])
    model.U = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['U'])

    # -------------------------------------------------------------------------
    # 3. Variables
    # -------------------------------------------------------------------------
    # x[i,j]: Continuous flow, non-negative
    model.x = pyo.Var(model.ARCS, domain=pyo.NonNegativeReals)
    # y[i,j]: Binary open/close decision
    model.y = pyo.Var(model.ARCS, domain=pyo.Binary)
    # s[i,j]: Slack variable for capacity constraints
    model.s = pyo.Var(model.ARCS, domain=pyo.NonNegativeReals)

    # -------------------------------------------------------------------------
    # 4. Objective Function
    # -------------------------------------------------------------------------
    # Minimize Total Cost = Sum(Fixed Costs + Variable Costs)
    def obj_rule(m):
        return sum(m.f[i, j] * m.y[i, j] + m.c[i, j] * m.x[i, j] for (i, j) in m.ARCS)
    model.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # -------------------------------------------------------------------------
    # 5. Constraints
    # -------------------------------------------------------------------------

    # Flow Balance Constraints 
    # sum(Inflow) - sum(Outflow) = Net Demand (b_i)
    #   For Supply node S: In(0) - Out = -40  => Out = 40
    #   For Demand node D: In - Out(0) = 25   => In = 25
    def flow_balance_rule(m, n):
        inflow = sum(m.x[i, n] for i in m.NODES if (i, n) in m.ARCS)
        outflow = sum(m.x[n, j] for j in m.NODES if (n, j) in m.ARCS)
        return inflow - outflow == m.b[n]
    model.FlowBalance = pyo.Constraint(model.NODES, rule=flow_balance_rule)

    # Capacity linking constraints
    # x[i,j] <= U[i,j] * y[i,j]
    def capacity_rule(m, i, j):
        return m.x[i, j] + m.s[i, j] == m.U[i, j] * m.y[i, j]
    model.CapacityConstraints = pyo.Constraint(model.ARCS, rule=capacity_rule)

    return model

def create_fcnf_relaxed_model():
    model = pyo.ConcreteModel(name="FCNF_Assignment3")

    # -------------------------------------------------------------------------
    # 1. Sets [cite: 139, 141]
    # -------------------------------------------------------------------------
    model.NODES = pyo.Set(initialize=['S', 'A', 'B', 'C', 'D', 'E'])
    model.ARCS = pyo.Set(dimen=2, initialize=[
        ('S', 'A'), ('S', 'B'), ('S', 'C'),
        ('A', 'B'), ('A', 'D'),
        ('B', 'C'), ('B', 'D'), ('B', 'E'),
        ('C', 'D'), ('C', 'E')
    ])

    # -------------------------------------------------------------------------
    # 2. Parameters [cite: 145-147, 151]
    # -------------------------------------------------------------------------
    # Net Supply/Demand (b_i): In - Out = b_i
    # Note: The problem defines positive b_i as demand and negative as supply.
    # S: -40, D: 25, E: 15, Others: 0
    b_data = {
        'S': -40,
        'A': 0,
        'B': 0,
        'C': 0,
        'D': 25,
        'E': 15
    }
    model.b = pyo.Param(model.NODES, initialize=b_data)

    # Arc Data: Variable Cost (c), Fixed Cost (f), Capacity (U)
    # Extracted from Table 1 
    arc_data = {
        ('S', 'A'): {'c': 2, 'f': 30, 'U': 25},
        ('S', 'B'): {'c': 3, 'f': 18, 'U': 20},
        ('S', 'C'): {'c': 4, 'f': 10, 'U': 20},
        ('A', 'B'): {'c': 1, 'f': 8,  'U': 15},
        ('B', 'C'): {'c': 1, 'f': 6,  'U': 15},
        ('A', 'D'): {'c': 3, 'f': 20, 'U': 20},
        ('B', 'D'): {'c': 2, 'f': 16, 'U': 25},
        ('B', 'E'): {'c': 2, 'f': 14, 'U': 15},
        ('C', 'D'): {'c': 1, 'f': 12, 'U': 15},
        ('C', 'E'): {'c': 3, 'f': 5,  'U': 20},
    }

    model.c = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['c'])
    model.f = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['f'])
    model.U = pyo.Param(model.ARCS, initialize=lambda m, i, j: arc_data[(i, j)]['U'])

    # -------------------------------------------------------------------------
    # 3. Variables
    # -------------------------------------------------------------------------
    # x[i,j]: Continuous flow, non-negative
    model.x = pyo.Var(model.ARCS, domain=pyo.NonNegativeReals)
    # y[i,j]: Binary open/close decision
    model.y = pyo.Var(model.ARCS, domain=pyo.NonNegativeReals, bounds=(0,1))  # Relaxed to [0,1]
    # s[i,j]: Slack variable for capacity constraints
    model.s = pyo.Var(model.ARCS, domain=pyo.NonNegativeReals)

    # -------------------------------------------------------------------------
    # 4. Objective Function
    # -------------------------------------------------------------------------
    # Minimize Total Cost = Sum(Fixed Costs + Variable Costs)
    def obj_rule(m):
        return sum(m.f[i, j] * m.y[i, j] + m.c[i, j] * m.x[i, j] for (i, j) in m.ARCS)
    model.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # -------------------------------------------------------------------------
    # 5. Constraints
    # -------------------------------------------------------------------------

    # Flow Balance Constraints 
    # sum(Inflow) - sum(Outflow) = Net Demand (b_i)
    #   For Supply node S: In(0) - Out = -40  => Out = 40
    #   For Demand node D: In - Out(0) = 25   => In = 25
    def flow_balance_rule(m, n):
        inflow = sum(m.x[i, n] for i in m.NODES if (i, n) in m.ARCS)
        outflow = sum(m.x[n, j] for j in m.NODES if (n, j) in m.ARCS)
        return inflow - outflow == m.b[n]
    model.FlowBalance = pyo.Constraint(model.NODES, rule=flow_balance_rule)

    # Capacity linking constraints
    # x[i,j] <= U[i,j] * y[i,j]
    def capacity_rule(m, i, j):
        return m.x[i, j] + m.s[i, j] == m.U[i, j] * m.y[i, j]
    model.CapacityConstraints = pyo.Constraint(model.ARCS, rule=capacity_rule)

    return model

In [ ]:
import pyomo.environ as pyo
import numpy as np

def solve_manual_gomory(model):
    # 1. Setup Solver with Presolve OFF (Crucial for Tableau access)
    opt = pyo.SolverFactory('gurobi_persistent')
    opt.set_instance(model)
    opt.set_gurobi_param('PreCrush', 1) 
    opt.set_gurobi_param('Presolve', 0) 
    
    if not hasattr(model, 'cuts'):
        model.cuts = pyo.ConstraintList()

    for iter_k in range(11): # Stop after 10 cuts
        results = opt.solve()
        if results.solver.termination_condition != pyo.TerminationCondition.optimal:
            print("Solver did not find optimal solution.")
            break
            
        g_model = opt._solver_model
        
        # --- A. Map Columns to Pyomo Variables & Types ---
        # We need to know if a column is Integer (y) or Continuous (x, s)
        col_to_info = {}
        for pyo_var, solver_var in opt._pyomo_var_to_solver_var_map.items():
            is_integer = pyo_var.parent_component().name == 'y'
            col_to_info[solver_var.index] = {'var': pyo_var, 'is_int': is_integer}

        # --- B. Find a Fractional Integer Variable ---
        target_var, target_col_idx = None, -1
        
        # Iterate through all variables in the Gurobi model
        for v in g_model.getVars():
            # Check if it corresponds to a Pyomo Integer variable
            if v.index in col_to_info and col_to_info[v.index]['is_int']:
                # Check if fractional
                if 1e-5 < v.X < 1 - 1e-5:
                    target_var = v
                    target_col_idx = v.index
                    break
        
        if target_var is None:
            print("Optimal Integer Solution found!")
            break

        print(f"Iteration {iter_k}: Generating cut for {target_var.VarName} = {target_var.X:.4f}")

        # --- C. Extract Basis and Invert (Equation 1) ---
        # Get A matrix and RHS b
        A = g_model.getA() # Returns scipy.sparse.csr_matrix
        b = np.array(g_model.getAttr("RHS"))
        
        # Identify Basic Columns (VBasis=0)
        vbasis = np.array(g_model.getAttr("VBasis", g_model.getVars()))
        basis_cols = np.where(vbasis == 0)[0]
        
        # Handle Rank Deficiency (Network Flow Redundancy)
        m, n = A.shape
        if len(basis_cols) < m:
            # Pad basis with non-basic columns if strictly needed for dimensions
            rem = [j for j in range(n) if j not in basis_cols]
            basis_cols = np.concatenate((basis_cols, rem[:m-len(basis_cols)]))
        
        # Extract B matrix
        B = A[:, basis_cols].toarray()
        
        try:
            # Use Pseudo-Inverse (pinv) to handle singular B in Network Flow
            B_inv = np.linalg.pinv(B)
            
            # Find the row index of our target variable within the Basis
            row_idx_in_basis = list(basis_cols).index(target_col_idx)
        except (ValueError, np.linalg.LinAlgError):
            print("  Error: Could not invert basis. Skipping.")
            break

        # Get Simplex Multipliers (pi) for this specific row
        # pi = (B^-1)_row
        pi = B_inv[row_idx_in_basis, :]

        # --- D. Calculate Tableau Row (Standard Form) ---
        # \bar{a} = pi * A
        # \bar{b} = pi * b
        bar_a = pi @ A
        bar_b = pi @ b
        
        f_0 = bar_b - np.floor(bar_b)
        
        # Safety check: if f_0 is too close to 0 or 1, we can't cut
        if f_0 < 1e-5 or f_0 > 1 - 1e-5:
            print("  RHS is effectively integer. Skipping.")
            continue

        # --- E. Generate GMI Cut (Formula from Source 80) ---
        # Formula: sum( coeff * x_j ) >= 1
        cut_expr = 0
        
        for j in range(n):
            if j not in col_to_info: continue # Skip internal Gurobi vars if any
            
            pyo_var = col_to_info[j]['var']
            is_int = col_to_info[j]['is_int']
            status = vbasis[j] # 0: Basic, -1: Lower, -2: Upper
            
            # Skip basic variables (except target, but target has f_j=0 so term=0)
            if status == 0: 
                continue

            aj = bar_a[j]
            
            # Handle variables at Upper Bound 
            
            if status == -2: # Non-basic at Upper Bound # Needed as the standard simplex assume it needs to be zero
                aj = -aj
                # We will use (UB - var) in the cut expression
                lb, ub = pyo_var.bounds
                var_term = ub - pyo_var
            else:
                var_term = pyo_var

            # Calculate Cut Coefficient
            term = 0
            
            if is_int:
                # Integer Logic
                fj = aj - np.floor(aj)
                if fj <= f_0:
                    term = fj / f_0
                else:
                    term = (1 - fj) / (1 - f_0)
            else:
                # Continuous Logic (x, s)
                if aj >= 0:
                    term = aj / f_0
                else:
                    term = -aj / (1 - f_0)
            
            # Add to expression if coefficient is significant
            cut_expr += term * var_term

        # --- F. Add Cut ---
        # Check if cut_expr is not 0 (it's a Pyomo expression)
        # We can check if it has terms.
        if hasattr(cut_expr, 'nargs') and cut_expr.nargs() > 0:
             # Explicitly says the RHS is >= 1
            c = model.cuts.add(cut_expr >= 1)
            opt.add_constraint(c)
            print("  Cut added successfully.")

        elif not isinstance(cut_expr, int) and not isinstance(cut_expr, float):
             # It might be a simple expression with 1 arg
            c = model.cuts.add(cut_expr >= 1)
            opt.add_constraint(c)
            print("  Cut added successfully.")

        else:
            print("  No valid cut terms found.")
            break

    return model

In [573]:
# Create the model
#model = create_fcnf_model() # OPTIMAL IS 285

model = create_fcnf_relaxed_model() # Relaxed found 265.7

# Example of how to solve if a solver is available (e.g., Gurobi, CBC, GLPK)
solver = pyo.SolverFactory('gurobi_persistent') 
solver.set_instance(model)
results = solver.solve(tee=True)
#model.display()

Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Xeon(R) CPU E5-2690 v4 @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 28 logical processors, using up to 28 threads

Academic license 2707350 - for non-commercial use only - registered to an___@kth.se
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Xeon(R) CPU E5-2690 v4 @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 28 logical processors, using up to 28 threads

Academic license 2707350 - for non-commercial use only - registered to an___@kth.se
Optimize a model with 16 rows, 30 columns and 50 nonzeros
Model fingerprint: 0x4759cf52
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 4e+01]
Presolve removed 11 rows and 21 columns
Presolve t

In [574]:
m = create_fcnf_relaxed_model()
m = solve_manual_gomory(m)
print("\nResults:")
m.display()

Set parameter PreCrush to value 1
Set parameter Presolve to value 0
Set parameter Presolve to value 0


Iteration 0: Generating cut for x22 = 0.2500
  Cut added successfully.
Iteration 1: Generating cut for x24 = 0.5781
  Cut added successfully.
Iteration 1: Generating cut for x24 = 0.5781
  Cut added successfully.
Iteration 2: Generating cut for x21 = 0.7029
  Cut added successfully.
Iteration 2: Generating cut for x21 = 0.7029
  Cut added successfully.
Iteration 3: Generating cut for x21 = 0.6990
  Cut added successfully.
Iteration 3: Generating cut for x21 = 0.6990
  Cut added successfully.
Iteration 4: Generating cut for x21 = 0.1090
  Cut added successfully.
Iteration 4: Generating cut for x21 = 0.1090
  Cut added successfully.
model.name="FCNF_Assignment3";
    - termination condition: infeasible
    - message from solver: <undefined>
Solver did not find optimal solution.

Results:
Model FCNF_Assignment3

  Variables:
    x : Size=10, Index=ARCS
        Key        : Lower : Value              : Upper : Fixed : Stale : Domain
        ('A', 'B') :     0 :  1.634956371600315 :  None :